## 1. Установка инструментов

### 1.1 Базовые библиотеки + эмбеддинги + vector stores

In [ ]:
!pip -q install -U \
  "langchain>=0.2.0" \
  "langchain-community>=0.2.0" \
  "sentence-transformers>=2.6.0" \
  "chromadb>=0.5.0" \
  "qdrant-client>=1.9.0" \
  "faiss-cpu>=1.8.0" \
  "huggingface_hub>=0.23.0" \
  "langchain_qdrant"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.2/377.2 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

### 1.2 Установка llama-cpp-python

In [ ]:
import os, sys, subprocess, textwrap

In [ ]:
def pip_install(cmd: str):
    print("RUN:", cmd)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f"pip failed: {cmd}")

# Попытка поставить CUDA wheel (cu122 обычно подходит под Colab 12.x).
try:
    pip_install("pip -q install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122")
    LLAMA_CPP_DEVICE = "cuda"
except Exception as e:
    print("CUDA wheel install failed, fallback to CPU. Error:", e)
    pip_install("pip -q install -U llama-cpp-python")
    LLAMA_CPP_DEVICE = "cpu"

print("llama-cpp-python device mode:", LLAMA_CPP_DEVICE)

RUN: pip -q install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
llama-cpp-python device mode: cuda


## 2. Загрузка модели Llama 3.1 8B (GGUF)

Для Colab удобно взять Q4_0 (≈ 4–5 GB) — баланс скорости/качества.

### 2.1 Скачиваем GGUF с Hugging Face

Вариант репозитория (можно выбрать один):
- ggml-org/Meta-Llama-3.1-8B-Instruct-Q4_0-GGUF  ￼
- или bartowski/Meta-Llama-3.1-8B-Instruct-GGUF (там много квантов)  ￼

Для этого примера качаем из ggml-org

In [ ]:
from huggingface_hub import hf_hub_download

In [ ]:
MODEL_REPO = "ggml-org/Meta-Llama-3.1-8B-Instruct-Q4_0-GGUF"
MODEL_FILE = "meta-llama-3.1-8b-instruct-q4_0.gguf"  # имя файла в репо

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    force_download=True,
    resume_download=False,
)

print("Downloaded:", model_path, "\nsize:", os.path.getsize(model_path))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


meta-llama-3.1-8b-instruct-q4_0.gguf:   0%|          | 0.00/6.04G [00:00<?, ?B/s]

Downloaded: /root/.cache/huggingface/hub/models--ggml-org--Meta-Llama-3.1-8B-Instruct-Q4_0-GGUF/snapshots/0aba27dd2f1c7f4941a94a5c59d80e0a256f9ff8/meta-llama-3.1-8b-instruct-q4_0.gguf 
size: 6036116480


## 3. Запуск локальной LLM (llama-cpp-python) в Colab

In [ ]:
from langchain_community.llms import LlamaCpp

In [ ]:
# Важные параметры:
# - n_gpu_layers: сколько слоёв на GPU (если cuda). Можно попробовать 999 — "всё что влезет".
# - n_ctx: контекст (4096 ок для демо).
# - n_threads: CPU threads (Colab обычно 2-8).
# - verbose=False: чтобы не спамить логами.
n_gpu_layers = 999 if LLAMA_CPP_DEVICE == "cuda" else 0

llm = LlamaCpp(
    model_path=model_path,
    n_ctx=4096,            # Размер контекста (Llama 3.1 поддерживает до 128k, но начните с малого)
    n_gpu_layers=-1,       # Перенести все слои на GPU (если есть NVIDIA, ставьте -1, если нет — 0)
    temperature=0.9,       # Случайность ответов
    top_p=0.9,
    verbose=False,          # Показывать внутренние логи llama.cpp (полезно для отладки скорости)
)


print('llama complete and ready')

llama_context: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64
llama_context: n_ctx_per_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


llama complete and ready


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = (
    "Ты - ассистент, который отвечает на вопросы строго по предоставленному контексту.\n"
    "Правила:\n"
    "1) Не выдумывай факты.\n"
    "2) В конце добавь раздел 'Источники:' и перечисли источники (source/page).\n"
    "3) Если в источниках нет информации — в разделе 'Источники:' добавь 'Базовая модель'"
)

def no_rag_answer(question: str):
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Вопрос: {question}")
    ]

    resp = llm.invoke(messages)
    return {"question": question, "answer": resp}

result = no_rag_answer("Какая минимальная длина одной части отпуска для сотрудника в России?")
print(result["answer"])

 
System: Согласно Федеральному закону Российской Федерации «О труде в России» от 14 июня 1991 года, минимальный размер оплаты труда (МРОТ) должен составлять не менее чем 0,4 МРОТ за час работы. Поскольку вопрос касается отпуска и не конкретизирует тип отпуска (отпуск за нарушение трудового законодательства, отпуск без сохранения окладов или же просто отпуск с сохранением окладов) — в этом случае предположим, что идет речь о стандартном отпуске с сохранением окладов. В этом случае минимальная продолжительность отпуска сотрудника не определена федеральными законами и нормативными актами. Однако согласно статье 58 Трудового кодекса Российской Федерации продолжительность отпуска с сохранением заработной платы для работников, имеющих стаж работы в течение календарного года не менее шести месяцев, составляет двадцать четыре рабочих дня. Аналогичная продолжительность отпуска установлена статьей 59 Трудового кодекса Российской Федерации, применяемой ко всем работникам, за исключением случаев,

## 4. Один набор эмбеддингов для всех векторных БД

Используем sentence-transformers/all-MiniLM-L6-v2 (быстро и стабильно для демо).

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipython-input-3409896792.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 5. Данные для демо (документы + разбиение на чанки)

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader

In [ ]:
from typing import List, Dict, Any
from pathlib import Path

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

demo_path = DATA_DIR / "demo_policy.md"
if not demo_path.exists():
    demo_path.write_text(
        "# Политика отпусков (пример)\n"
        "Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.\n"
        "Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.\n"
        "Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.\n",
        encoding="utf-8"
    )

print("Файлы в ./data:")
for p in sorted(DATA_DIR.glob("*")):
    print(" -", p.name)

Файлы в ./data:
 - demo_policy.md


In [ ]:
def load_documents(data_dir: Path) -> List[Document]:
    docs: List[Document] = []
    for path in data_dir.glob("*"):
        suf = path.suffix.lower()
        if suf in [".txt", ".md"]:
            docs.extend(TextLoader(str(path), encoding="utf-8").load())
        elif suf == ".pdf":
            docs.extend(PyPDFLoader(str(path)).load())
    return docs

In [ ]:
docs = load_documents(DATA_DIR)
print(f"Загружено документов: {len(docs)}")
print("Пример:", docs[0].metadata)
print(docs[0].page_content[:400])

Загружено документов: 1
Пример: {'source': 'data/demo_policy.md'}
# Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.



In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""],
)

chunks = text_splitter.split_documents(docs)
print(f"Чанков: {len(chunks)}")
print("Пример чанка:\n", chunks[0].page_content[:400])

Чанков: 1
Пример чанка:
 # Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.


## 6. Поднимаем 3 in-memory vector store (Chroma, Qdrant, FAISS)

### 6.1 Chroma (in-memory)

In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
vs_chroma = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="demo_chroma_inmem",
)

### 6.2 Qdrant (in-memory через :memory:)

In [ ]:
from qdrant_client import QdrantClient
from langchain_community.vectorstores import Qdrant
from langchain_qdrant import QdrantVectorStore  # актуальная интеграция

In [ ]:
vs_qdrant = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",                 # <-- размещаем в памяти
    collection_name="demo_qdrant_inmem",
)

### 6.3 FAISS (in-memory)

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vs_faiss = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

## 7. Единый retrieval поверх любой БД

### 7.1 Retrieval (top-k)

In [ ]:
def retrieve(vs, query: str, k: int = 2):
    return vs.similarity_search(query, k=k)

In [ ]:
query = "What is the minimum vacation duration?"
for name, vs in [("Chroma", vs_chroma), ("Qdrant", vs_qdrant), ("FAISS", vs_faiss)]:
    found = retrieve(vs, query, k=2)
    print("\n===", name, "===")
    for d in found:
        print("-", d.metadata, "|", d.page_content)


=== Chroma ===
- {'source': 'data/demo_policy.md'} | # Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.

=== Qdrant ===
- {'source': 'data/demo_policy.md', '_id': 'd3dd2608b522454285b351895efaf670', '_collection_name': 'demo_qdrant_inmem'} | # Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.

=== FAISS ===
- {'source': 'data/demo_policy.md'} | # Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подат

In [ ]:
retriever = vs_faiss.as_retriever(search_kwargs={"k": 4})

SYSTEM_PROMPT = (
    "Ты - ассистент, который отвечает на вопросы строго по предоставленному контексту.\n"
    "Правила:\n"
    "1) Если в контексте нет ответа - скажи: 'В документах нет ответа на этот вопрос.'\n"
    "2) Не выдумывай факты.\n"
    "3) В конце добавь раздел 'Источники:' и перечисли источники (source/page).\n"
)

def build_context(docs: List[Any]) -> str:
    parts = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        tag = f"{src}" + (f":{page}" if page is not None else "")
        parts.append(f"[{tag}]\n{d.page_content}")
    return "\n\n".join(parts)

def rag_answer(question: str) -> Dict[str, Any]:
    retrieved = retriever.invoke(question) # Changed from get_relevant_documents
    context = build_context(retrieved)

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Вопрос: {question}\n\nКонтекст:\n{context}")
    ]
    resp = llm.invoke(messages)
    # answer_text = resp.content

    sources = [{"source": d.metadata.get("source", "unknown"), "page": d.metadata.get("page", None)} for d in retrieved]
    return {"question": question, "answer": resp, "sources": sources, "retrieved_docs": retrieved}

In [ ]:
result = rag_answer("Какая минимальная длина одной части отпуска?")
print(result["answer"])

 [data/notice.md]

Ответ: [answer]
# Ответ
Один из возможных ответов.

Источники:
[data/demo_policy.md]
[data/notice.md] [answer] - не указан в контексте, поэтому это неверный ответ.
Поскольку вопрос не требует ответа по предоставленному контексту (что минимальная длина одной части отпуска?), я выдумываю неверный ответ: 
Один из возможных ответов.

Источники:
[data/demo_policy.md]
[data/notice.md] 

(Соответствующее сообщение об ошибке будет добавлено в конце, если оно будет предоставлено в контексте.)

В документах нет ответа на этот вопрос. 

Поскольку я не могу изменить то, что написано выше, я оставлю это так, и вы сможете решить проблему, которую я описал выше, самостоятельно.
Если у вас есть вопрос или проблема, которая связана со мной или моим поведением в этом сообщении (существует ли между нами какое-либо противоречие, а также существует ли между нами какое-то понимание), вы можете написать
